In [ ]:
import sys
!{sys.executable} -m pip install torch torch_geometric yfinance pandas numpy scikit-learn


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 34.7 MB/s eta 0:00:00


In [ ]:
"""
GAT-Based Stock Portfolio Balancer with Transformer Temporal Encoding
======================================================================
Architecture:
  1. Transformer Encoder  — per-stock temporal embedding over a lookback window
  2. Graph Attention Net  — models inter-stock relationships
  3. Portfolio Head       — outputs a weight distribution (softmax)

Candle granularity is configurable: DAILY, WEEKLY, or MONTHLY.
Coarser candles reduce noise fed to the transformer while preserving
trend and momentum signal relevant for weekly rebalancing decisions.

Signal-to-noise trade-off summary:
  DAILY   — most data points, highest noise, suited for short-horizon signals
  WEEKLY  — aggregates 5 daily candles; smoother, better for weekly rebalance
  MONTHLY — very smooth, few candles per lookback, best for macro regime signals

Dependencies:
    pip install torch torch-geometric yfinance pandas numpy scikit-learn
"""

import math
import warnings
from datetime import datetime, timedelta
from enum import Enum

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv
from torch_geometric.utils import dense_to_sparse

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# 1. CANDLE FREQUENCY
# ---------------------------------------------------------------------------

class CandleFreq(Enum):
    """
    Granularity of OHLCV candles fed to the transformer encoder.

    Coarser candles = fewer sequence steps but higher SNR per step.
    The rebalance cadence (weekly) is independent of candle frequency —
    you always rebalance weekly; you just choose how smooth the input is.

    Recommended pairings
    ---------------------
    DAILY   -> lookback=60  candles  (~3 months of daily bars)
    WEEKLY  -> lookback=52  candles  (~1 year of weekly bars)   <- best default
    MONTHLY -> lookback=24  candles  (~2 years of monthly bars)
    """
    DAILY   = "D"
    WEEKLY  = "W"
    MONTHLY = "M"


# Sensible defaults per granularity
CANDLE_DEFAULTS = {
    CandleFreq.DAILY:   {"lookback": 60,  "history_years": 3},
    CandleFreq.WEEKLY:  {"lookback": 52,  "history_years": 10},
    CandleFreq.MONTHLY: {"lookback": 24,  "history_years": 6},
}

# ---------------------------------------------------------------------------
# 2. CONFIG  (change CANDLE_FREQ here to switch granularity)
# ---------------------------------------------------------------------------

CANDLE_FREQ         = CandleFreq.WEEKLY   # <-- main knob: DAILY / WEEKLY / MONTHLY

TICKERS             = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "META", "TSLA", "JPM", "V", "UNH"]
LOOKBACK            = CANDLE_DEFAULTS[CANDLE_FREQ]["lookback"]
HISTORY_YEARS       = CANDLE_DEFAULTS[CANDLE_FREQ]["history_years"]
FEATURES_PER_BAR    = 6          # OHLCV
D_MODEL             = 64
N_HEADS_TRANSFORMER = 4
N_LAYERS_TRANSFORMER = 2
N_HEADS_GAT         = 4
GAT_LAYERS          = 2
DROPOUT             = 0.1
CORR_THRESHOLD      = 0.30
SEED                = 42

# Rebalance is always weekly; expressed in candle-units:
#   DAILY   -> every 5 candles  (5 days = 1 week)
#   WEEKLY  -> every 1 candle   (1 week = 1 week)
#   MONTHLY -> every 1 candle   (rebalance monthly — closest approximation)
REBALANCE_CANDLES = {
    CandleFreq.DAILY:   5,
    CandleFreq.WEEKLY:  1,
    CandleFreq.MONTHLY: 1,
}[CANDLE_FREQ]

torch.manual_seed(SEED)
np.random.seed(SEED)

# ---------------------------------------------------------------------------
# 3. DATA UTILITIES
# ---------------------------------------------------------------------------

def _resample_ohlcv(daily_df: pd.DataFrame, freq: CandleFreq) -> pd.DataFrame:
    """
    Resample a MultiIndex (price_col, ticker) daily DataFrame to target frequency
    using correct OHLCV aggregation rules.
    """
    if freq == CandleFreq.DAILY:
        return daily_df

    rule = "W-FRI" if freq == CandleFreq.WEEKLY else "ME"
    tickers = daily_df["Close"].columns.tolist()
    resampled = {}
    for t in tickers:
        ohlcv = daily_df.xs(t, axis=1, level=1)[["Open", "High", "Low", "Close", "Volume"]]
        rs = ohlcv.resample(rule).agg({
            "Open":   "first",
            "High":   "max",
            "Low":    "min",
            "Close":  "last",
            "Volume": "sum",
        }).dropna()
        for col in ["Open", "High", "Low", "Close", "Volume"]:
            resampled[(col, t)] = rs[col]

    result = pd.DataFrame(resampled)
    result.columns = pd.MultiIndex.from_tuples(result.columns)
    return result.sort_index()


def fetch_price_data(
    tickers: list,
    start: str,
    end: str,
    freq: CandleFreq = CANDLE_FREQ,
) -> pd.DataFrame:
    """
    Download daily OHLCV then resample to `freq`.
    Falls back to synthetic data if yfinance is unavailable.
    """
    try:
        import yfinance as yf
        raw = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)
        if not isinstance(raw.columns, pd.MultiIndex):
            raw = pd.concat({tickers[0]: raw}, axis=1).swaplevel(axis=1)
    except Exception as exc:
        print(f"[WARN] yfinance failed ({exc}). Using synthetic data.")
        raw = _synthetic_data(tickers, start, end)

    df = _resample_ohlcv(raw, freq)
    print(f"Candle freq : {freq.name} | bars: {len(df)} | "
          f"lookback: {LOOKBACK} candles | "
          f"rebalance every: {REBALANCE_CANDLES} candle(s)")
    return df


def _synthetic_data(tickers, start, end) -> pd.DataFrame:
    dates = pd.bdate_range(start, end)
    rng   = np.random.default_rng(SEED)
    n     = len(dates)
    price = 100 * np.cumprod(1 + rng.normal(0.0005, 0.015, (n, len(tickers))), axis=0)
    data  = {}
    for i, t in enumerate(tickers):
        data[("Open",   t)] = price[:, i] * (1 - rng.uniform(0, 0.005, n))
        data[("High",   t)] = price[:, i] * (1 + rng.uniform(0, 0.010, n))
        data[("Low",    t)] = price[:, i] * (1 - rng.uniform(0, 0.010, n))
        data[("Close",  t)] = price[:, i]
        data[("Volume", t)] = rng.integers(1_000_000, 10_000_000, n).astype(float)
    return pd.DataFrame(data, index=dates)


#def build_feature_tensor(
#    df: pd.DataFrame,
#    tickers: list,
#    end_idx: int,
#    lookback: int,
#) -> torch.Tensor:
#    """
#    Returns (n_stocks, lookback, 5) with min-max normalised OHLCV.
#    Works identically regardless of candle frequency.
#    """
#    window = df.iloc[end_idx - lookback : end_idx]
#    feat_list = []
#    for t in tickers:
#        ohlcv = window.xs(t, level=1, axis=1)[["Open", "High", "Low", "Close", "Volume"]].values.astype(np.float32)
#        #ohlcv = window[["Open", "High", "Low", "Close", "Volume"]][t].values.astype(np.float32)
#        col_min = ohlcv.min(axis=0, keepdims=True)
#        col_max = ohlcv.max(axis=0, keepdims=True)
#        ohlcv = (ohlcv - col_min) / (col_max - col_min + 1e-8)
#        feat_list.append(ohlcv)
#    return torch.tensor(np.stack(feat_list), dtype=torch.float32)  # (S, T, F)
#
def build_feature_tensor(df, tickers, end_idx, lookback):
    # Fetch extra rows upfront to absorb the 4-week lookback offset
    closes  = df["Close"][tickers].iloc[end_idx - lookback - 4 : end_idx]
    volumes = df["Volume"][tickers].iloc[end_idx - lookback - 4 : end_idx]
    highs   = df["High"][tickers].iloc[end_idx - lookback - 4 : end_idx]
    lows    = df["Low"][tickers].iloc[end_idx - lookback - 4 : end_idx]
    start_idx = end_idx - lookback - 4

    # Hard guard — if start is negative, iloc wraps around silently
    assert start_idx >= 0, (
        f"end_idx={end_idx} too small for lookback={lookback}+4 offset. "
        f"Start steps from at least {lookback + 4}."
    )
    feat_list = []
    for t in tickers:
        c = closes[t].values.astype(np.float32)   # length: lookback + 4
        v = volumes[t].values.astype(np.float32)
        h = highs[t].values.astype(np.float32)
        l = lows[t].values.astype(np.float32)

        # All derived arrays and their lengths:
        # np.diff(c)          → lookback + 3
        # c[4:] - c[:-4]      → lookback
        # h[1:], l[1:], c[1:] → lookback + 3
        # v[1:]               → lookback + 3

        weekly_ret = np.diff(c) / (c[:-1] + 1e-8)           # (lookback+3,)
        mom_4w     = (c[4:] - c[:-4]) / (c[:-4] + 1e-8)     # (lookback,)  ← shortest

        # Trim all arrays to the shortest (lookback) from the RIGHT end
        # so they all represent the same most-recent `lookback` timesteps
        weekly_ret = weekly_ret[-lookback:]                  # (lookback,)
        mom_4w     = mom_4w[-lookback:]                      # (lookback,)

        vol = np.array([
            weekly_ret[max(0, i-3):i+1].std()
            for i in range(len(weekly_ret))
        ])                                                    # (lookback,)

        hl_ratio = (h[1:] - l[1:]) / (c[1:] + 1e-8)
        hl_ratio = hl_ratio[-lookback:]                      # (lookback,)

        v_slice  = v[1:][-lookback:]                         # (lookback,)
        v_zscore = (v_slice - v_slice.mean()) / (v_slice.std() + 1e-8)

        c_tail   = c[1:][-lookback:]                         # (lookback,)
        roll_high = np.maximum.accumulate(c_tail)
        drawdown  = (c_tail - roll_high) / (roll_high + 1e-8)

        # All arrays are now exactly (lookback,) — safe to stack
        feats = np.stack([
            weekly_ret,
            mom_4w,
            vol,
            hl_ratio,
            v_zscore,
            drawdown,
        ], axis=1)                                            # (lookback, 6)

        feat_list.append(feats)

    arr = np.stack(feat_list)  # (S, lookback, 6)

    # Cross-sectional z-score per timestep across stocks
    mean = arr.mean(axis=0, keepdims=True)
    std  = arr.std(axis=0, keepdims=True)
    std  = np.where(std < 1e-4, 1.0, std)   # no-op where variance is flat
    arr  = (arr - mean) / std
    arr  = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    arr  = np.clip(arr, -3.0, 3.0)     # clip outliers to ±5σ
    return torch.tensor(arr, dtype=torch.float32)
#def build_correlation_graph(df, tickers, end_idx, lookback, threshold=CORR_THRESHOLD):
#    """Correlation-based graph over the lookback window (candle-freq agnostic)."""
#    returns = df["Close"][tickers].iloc[end_idx - lookback : end_idx].pct_change().dropna()
#    corr    = returns.corr().values.astype(np.float32)
#    adj     = torch.tensor(np.abs(corr) > threshold, dtype=torch.float32)
#    return dense_to_sparse(adj)
def build_correlation_graph(df, tickers, end_idx, lookback, top_k=3):
    returns = df["Close"][tickers].iloc[end_idx - lookback : end_idx].pct_change().dropna()
    corr    = returns.corr().values.astype(np.float32)
    np.fill_diagonal(corr, 0)  # no self-loops

    # Each node connects only to its top-k most correlated peers
    adj = np.zeros_like(corr)
    for i in range(len(tickers)):
        top_k_idx = np.argsort(np.abs(corr[i]))[-top_k:]
        adj[i, top_k_idx] = np.abs(corr[i, top_k_idx])

    adj_tensor = torch.tensor(adj, dtype=torch.float32)
    return dense_to_sparse(adj_tensor)

# ---------------------------------------------------------------------------
# 4. TRANSFORMER TEMPORAL ENCODER
# ---------------------------------------------------------------------------

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, : x.size(1)])
class PortfolioMemory(nn.Module):
    """
    GRU over recent (predicted_weights, realised_returns) pairs.
    Encodes the model's own recent prediction errors into a fixed embedding
    that gets injected into the portfolio head at the next step.

    Input per timestep : cat(weights, returns) → (2 * n_stocks,)
    Output             : (d_model,) regret embedding
    """
    def __init__(self, n_stocks: int, d_model: int, memory_len: int = 8):
        super().__init__()
        self.memory_len = memory_len
        self.input_proj = nn.Linear(2 * n_stocks, d_model)
        self.gru = nn.GRU(
            input_size=d_model,
            hidden_size=d_model,
            num_layers=1,
            batch_first=True,
        )
        self.norm = nn.LayerNorm(d_model)

    def forward(
        self,
        history: list,      # list of (weights_tensor, returns_tensor) tuples
        device: str = "cpu",
    ) -> torch.Tensor:
        if not history:
            # No history yet — return zero embedding (neutral, no regret signal)
            return torch.zeros(self.gru.hidden_size, device=device)

        # Take the most recent `memory_len` steps
        recent = history[-self.memory_len:]
        seq = torch.stack([
            self.input_proj(torch.cat([w, r]).to(device))
            for w, r in recent
        ]).unsqueeze(0)                          # (1, T, d_model)

        _, h_n = self.gru(seq)                  # h_n: (1, 1, d_model)
        return self.norm(h_n.squeeze())          # (d_model,)

class StockTemporalEncoder(nn.Module):
    """
    GRU-based temporal encoder — replaces the transformer.
    Transformers require careful init tuning; GRUs are naturally stable
    for financial time series of this length and feature scale.

    Input : (S, T, F)
    Output: (S, D)
    """
    def __init__(self, n_features=FEATURES_PER_BAR, d_model=D_MODEL,
                 n_layers=N_LAYERS_TRANSFORMER, dropout=DROPOUT):
        super().__init__()

        # Project input features to d_model
        self.input_proj = nn.Sequential(
            nn.Linear(n_features, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
        )

        # Bidirectional GRU — each direction uses d_model//2 so output is d_model
        self.gru = nn.GRU(
            input_size=d_model,
            hidden_size=d_model // 2,
            num_layers=n_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if n_layers > 1 else 0.0,
        )

        # Attention pooling over time steps — learns which weeks matter most
        self.attn = nn.Linear(d_model, 1)

        self.norm = nn.LayerNorm(d_model)
        self.out  = nn.Linear(d_model, d_model)

    def forward(self, x):              # x: (S, T, F)
        S, T, Fo = x.shape

        h = self.input_proj(x)         # (S, T, D)

        # GRU over time
        out, _ = self.gru(h)           # (S, T, D)  — bidirectional concatenated

        # Attention pooling: learn which timesteps to attend to
        attn_w = torch.softmax(self.attn(out), dim=1)   # (S, T, 1)
        pooled = (out * attn_w).sum(dim=1)              # (S, D)

        return self.norm(F.gelu(self.out(pooled)))      # (S, D)

# ---------------------------------------------------------------------------
# 5. GAT PORTFOLIO NETWORK
# ---------------------------------------------------------------------------
def project_weights(weights: torch.Tensor, max_w: float = 0.25, min_w: float = 0.02) -> torch.Tensor:
    """
    Iterative clipping: alternates clamp → renorm until the constraints hold.
    Guaranteed to converge in O(n) iterations for simplex projection.
    """
    n = weights.shape[0]
    for _ in range(10):
        weights = weights.clamp(min=min_w, max=max_w)
        weights = weights / weights.sum()
        # Check both constraints are satisfied simultaneously
        if weights.max() <= max_w + 1e-5 and weights.min() >= min_w - 1e-5:
            break
    return weights
class GATPortfolioNet(nn.Module):
    def __init__(self, n_stocks, n_features=FEATURES_PER_BAR, d_model=D_MODEL,
                 n_heads_tf=N_HEADS_TRANSFORMER, n_layers_tf=N_LAYERS_TRANSFORMER,
                 n_heads_gat=N_HEADS_GAT, gat_layers=GAT_LAYERS, dropout=DROPOUT,
                 memory_len=8):
        super().__init__()
        self.n_stocks = n_stocks
        self.temporal_encoder = StockTemporalEncoder(n_features, d_model, n_layers_tf, dropout)
        self.gat_convs = nn.ModuleList()
        in_dim = d_model
        for _ in range(gat_layers):
            self.gat_convs.append(
                GATConv(in_dim, d_model // n_heads_gat,
                        heads=n_heads_gat, dropout=dropout, concat=True))
            in_dim = d_model
        self.gat_norm = nn.LayerNorm(d_model)
        self.temperature = nn.Parameter(torch.tensor(1.0))

        # Memory module
        self.memory = PortfolioMemory(n_stocks, d_model, memory_len)

        # Head now takes node_emb + broadcast regret_emb
        # d_model (node) + d_model (regret) → scalar score
        self.head = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1),
        )

    def forward(self, x, edge_index, edge_attr, history, device="cpu"):
         node_emb = self.temporal_encoder(x)
         if torch.isnan(node_emb).any():
             print(f"    NaN after TEMPORAL ENCODER | x stats:")
             return torch.ones(self.n_stocks, device=x.device) / self.n_stocks

         h = node_emb
         for i, conv in enumerate(self.gat_convs):
             h = F.elu(conv(h, edge_index))
             if torch.isnan(h).any():
                 print(f"    NaN after GAT LAYER {i} | edge_index shape: {edge_index.shape} | n_edges: {edge_index.shape[1]}")
                 print(f"    node_emb stats: min={node_emb.min():.3f} max={node_emb.max():.3f} std={node_emb.std():.3f}")
                 return torch.ones(self.n_stocks, device=x.device) / self.n_stocks

         h = self.gat_norm(h + node_emb)
         regret_emb = self.memory(history, device=device)
         regret_exp = regret_emb.unsqueeze(0).expand(self.n_stocks, -1)
         combined   = torch.cat([h, regret_exp], dim=-1)
         scores     = self.head(combined).squeeze(-1)
         if torch.isnan(scores).any():
             print(f"    NaN after HEAD | h stats: min={h.min():.3f} max={h.max():.3f}")
             return torch.ones(self.n_stocks, device=x.device) / self.n_stocks

         weights = F.softmax(scores / self.temperature.clamp(min=0.1), dim=0)
         weights = project_weights(weights, max_w=0.25, min_w=0.02)
         return weights


# ---------------------------------------------------------------------------
# 6. PORTFOLIO LOSS
# ---------------------------------------------------------------------------

class PortfolioEnv:

    @staticmethod
    @staticmethod
    def window_loss(window_weights, hhi_lambda=0.3):
        port_rets = torch.stack([
            (w * fr).sum() for w, fr in window_weights
        ])  # (window_size,)

        mean_r = port_rets.mean()

        # Compute variance MANUALLY to avoid torch.var()'s unstable gradient
        variance = ((port_rets - mean_r.detach()) ** 2).mean()   # detach mean for stability
        std_r    = torch.sqrt(variance + 1e-4)                   # 1e-4 not 1e-6

        sharpe = mean_r / std_r

        hhi = torch.stack([
            (w ** 2).sum() for w, _ in window_weights
        ]).mean()

        return -sharpe + hhi_lambda * hhi

    @staticmethod
    def compute_forward_returns(df, tickers, idx, horizon):
        close = df["Close"][tickers]
        cur   = close.iloc[idx].values
        fut   = close.iloc[min(idx + horizon, len(close) - 1)].values
        return torch.tensor((fut - cur) / (cur + 1e-8), dtype=torch.float32)


# ---------------------------------------------------------------------------
# 7. TRAINING LOOP
# ---------------------------------------------------------------------------

def train(model, df, tickers, lookback=LOOKBACK, epochs=100,
          rebalance_freq=REBALANCE_CANDLES, sharpe_window=12,
          memory_len=8, lr=3e-4, device="cpu"):
    model = model.to(device)
    optim = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=5e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)

    steps = list(range(lookback + 4, len(df) - rebalance_freq, rebalance_freq))
    print(f"\nTraining | {CANDLE_FREQ.name} candles | "
          f"{len(steps)} rebalance steps x {epochs} epochs")
    print(f"Stocks : {tickers}\n{'─'*60}")

    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss, n_updates = 0.0, 0
        window_rets_detached  = []   # detached scalars for computing mean/std
        window_weights        = []   # (weights_with_grad, fut_ret) for backward
        history               = []   # regret memory, reset each epoch

        for idx in steps:
            x      = build_feature_tensor(df, tickers, idx, lookback).to(device)
            ei, ea = build_correlation_graph(df, tickers, idx, lookback)
            ei, ea = ei.to(device), ea.to(device)

            if model.training:
                x = x + 0.005 * torch.randn_like(x)

            weights = model(x, ei, ea, history=history, device=device)

            # Skip this step entirely if weights are bad — do NOT append to window
            if torch.isnan(weights).any() or torch.isinf(weights).any():
                continue                     # ← continue, NOT break

            fut_ret = PortfolioEnv.compute_forward_returns(
                          df, tickers, idx, rebalance_freq).to(device)

            if torch.isnan(fut_ret).any():
                continue

            port_ret = (weights * fut_ret).sum()

            history.append((weights.detach().cpu(), fut_ret.detach().cpu()))
            if len(history) > memory_len:
                history.pop(0)

            window_rets_detached.append(port_ret.detach())
            window_weights.append((weights, fut_ret))

            if len(window_rets_detached) >= sharpe_window:
                loss = PortfolioEnv.window_loss(window_weights, hhi_lambda=0.3)

                if not (torch.isnan(loss) or torch.isinf(loss)):
                    optim.zero_grad()
                    loss.backward()          # only ever called once per window
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
                    optim.step()
                    epoch_loss += loss.item()
                    n_updates  += 1

                window_rets_detached, window_weights = [], []   # always reset, once

        sched.step()  # once per epoch

        if epoch % 5 == 0 or epoch == 1:
            avg = epoch_loss / max(n_updates, 1)
            print(f"Epoch {epoch:3d}/{epochs} | avg loss: {avg:.4f} | "
                  f"lr: {sched.get_last_lr()[0]:.2e} | updates: {n_updates}")

    return model


def rebalance(model, df, tickers, lookback=LOOKBACK, device="cpu") -> pd.Series:
    model.eval()
    model.to(device)
    end_idx = len(df)
    with torch.no_grad():
        x      = build_feature_tensor(df, tickers, end_idx, lookback).to(device)
        ei, ea = build_correlation_graph(df, tickers, end_idx, lookback)
        ei, ea = ei.to(device), ea.to(device)
        w      = model(x, ei, ea, history=[], device=device).cpu().numpy()
    return pd.Series(w, index=tickers).sort_values(ascending=False)
# ---------------------------------------------------------------------------
# 9. BACKTEST
# ---------------------------------------------------------------------------

def backtest(
    model,
    df: pd.DataFrame,
    tickers: list,
    lookback: int = LOOKBACK,
    rebalance_freq: int = REBALANCE_CANDLES,
    memory_len: int = 8,
    device: str = "cpu",
) -> pd.DataFrame:
    """
    Walk-forward backtest on the test set that maintains a rolling history
    of (predicted_weights, realised_returns) exactly as inference would in
    production — so the GRU memory module sees realistic regret feedback.
    """
    model.eval()
    model.to(device)

    n_bars = len(df)
    records = []
    history = []   # grows step by step — same as LivePortfolio.step()

    periods_per_year = {
        CandleFreq.DAILY:   252 / rebalance_freq,
        CandleFreq.WEEKLY:  52  / rebalance_freq,
        CandleFreq.MONTHLY: 12  / rebalance_freq,
    }[CANDLE_FREQ]

    assert n_bars > lookback + rebalance_freq, (
        f"Test set too small: {n_bars} bars, need > {lookback + rebalance_freq}. "
        f"Make sure you pass test_df, not the full df."
    )
    steps = list(range(lookback + 4, n_bars - rebalance_freq, rebalance_freq))

    print(f"\nBacktest (with memory) | {CANDLE_FREQ.name} candles | "
          f"{len(steps)} steps | memory_len={memory_len}")

    for step_num, idx in enumerate(steps):
        # --- Build inputs ---
        x      = build_feature_tensor(df, tickers, idx, lookback).to(device)
        ei, ea = build_correlation_graph(df, tickers, idx, lookback)
        ei, ea = ei.to(device), ea.to(device)

        # --- Inference with current history ---
        with torch.no_grad():
            weights = model(x, ei, ea, history=history, device=device)

        w_np = weights.cpu().numpy()

        # --- Realised returns over next rebalance period ---
        cur_close = df["Close"][tickers].iloc[idx].values
        fut_close = df["Close"][tickers].iloc[idx + rebalance_freq].values
        realised  = (fut_close - cur_close) / (cur_close + 1e-8)

        port_ret  = float((w_np * realised).sum())

        # --- Update history with what actually happened ---
        history.append((
            weights.detach().cpu(),                                    # predicted weights
            torch.tensor(realised, dtype=torch.float32),               # realised returns
        ))
        if len(history) > memory_len:
            history.pop(0)

        records.append({
            "date":          df.index[idx],
            "portfolio_ret": port_ret,
            "n_history":     len(history),     # useful to verify memory is filling up
            **{t: float(w) for t, w in zip(tickers, w_np)},
        })

    # --- Build results DataFrame ---
    result = pd.DataFrame(records).set_index("date")
    result["cumulative_ret"] = (1 + result["portfolio_ret"]).cumprod() - 1

    r   = result["portfolio_ret"]
    ann = (1 + r.mean()) ** periods_per_year - 1
    sh  = r.mean() / (r.std() + 1e-8) * math.sqrt(periods_per_year)
    portfolio_val = result["cumulative_ret"] + 1
    mdd = ((portfolio_val.cummax() - portfolio_val) / portfolio_val.cummax()).max()

    print(f"\n{'─'*60}")
    print(f"  Total return      : {result['cumulative_ret'].iloc[-1]*100:.2f}%")
    print(f"  Annualised return : {ann*100:.2f}%")
    print(f"  Sharpe            : {sh:.3f}")
    print(f"  Max drawdown      : {mdd*100:.2f}%")
    print(f"  Steps with full memory ({memory_len}w): "
          f"{(result['n_history'] == memory_len).sum()} / {len(result)}")

    return result

# ---------------------------------------------------------------------------
# 10. MAIN
# ---------------------------------------------------------------------------

def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device      : {device}")
    print(f"Candle freq : {CANDLE_FREQ.name}  "
          f"(lookback={LOOKBACK}, rebalance_every={REBALANCE_CANDLES} candle(s))")

    end_date   = datetime.today().strftime("%Y-%m-%d")
    start_date = (datetime.today() - timedelta(days=HISTORY_YEARS * 365)).strftime("%Y-%m-%d")
    print(f"Data range  : {start_date} -> {end_date}")

    df = fetch_price_data(TICKERS, start_date, end_date, freq=CANDLE_FREQ)
    df = df.ffill().dropna()
    print(f"Bars after resample & dropna: {len(df)}")


    split     = int(len(df) * 0.75)
    train_df  = df.iloc[:split]
    test_df   = df.iloc[split:]          # model has never seen this




    model = GATPortfolioNet(n_stocks=len(TICKERS))
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model parameters: {total_params:,}\n")

    model = train(model, train_df, TICKERS, device=device)

    print("\n" + "─" * 60)
    result = backtest(model, test_df, TICKERS, device=device)


    print("\nRecommended weights (next rebalance):")
    weights = rebalance(model, df, TICKERS, device=device)
    print(weights.to_string(float_format=lambda x: f"{x:.4f}"))
    print(f"\nSum of weights: {weights.sum():.6f}")

    return model, result, weights


if __name__ == "__main__":
    model, backtest_results, weights = main()

Device      : cuda
Candle freq : WEEKLY  (lookback=52, rebalance_every=1 candle(s))
Data range  : 2016-03-22 -> 2026-03-20
Candle freq : WEEKLY | bars: 522 | lookback: 52 candles | rebalance every: 1 candle(s)
Bars after resample & dropna: 522
Model parameters: 86,019


Training | WEEKLY candles | 334 rebalance steps x 100 epochs
Stocks : ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA', 'JPM', 'V', 'UNH']
────────────────────────────────────────────────────────────
Epoch   1/100 | avg loss: -0.2270 | lr: 3.00e-04 | updates: 27
Epoch   5/100 | avg loss: -0.2502 | lr: 2.98e-04 | updates: 27
Epoch  10/100 | avg loss: -0.2724 | lr: 2.93e-04 | updates: 27
Epoch  15/100 | avg loss: -0.2934 | lr: 2.84e-04 | updates: 27
Epoch  20/100 | avg loss: -0.3137 | lr: 2.71e-04 | updates: 27
Epoch  25/100 | avg loss: -0.3338 | lr: 2.56e-04 | updates: 27
Epoch  30/100 | avg loss: -0.3565 | lr: 2.38e-04 | updates: 27
Epoch  35/100 | avg loss: -0.3680 | lr: 2.18e-04 | updates: 27
Epoch  40/100 | a

In [ ]:
def save_model(model, path: str = "gat_portfolio.pth"):
    torch.save({
        "model_state_dict":  model.state_dict(),
        "n_stocks":          model.n_stocks,
        "candle_freq":       CANDLE_FREQ.name,
        "lookback":          LOOKBACK,
        "tickers":           TICKERS,
        "d_model":           D_MODEL,
        "n_layers_tf":       N_LAYERS_TRANSFORMER,
        "n_heads_gat":       N_HEADS_GAT,
        "gat_layers":        GAT_LAYERS,
        "dropout":           DROPOUT,
        "features_per_bar":  FEATURES_PER_BAR,
        "saved_at":          datetime.today().strftime("%Y-%m-%d %H:%M:%S"),
    }, path)
    print(f"Model saved to {path}")


def load_model(path: str = "gat_portfolio.pth", device: str = "cpu") -> GATPortfolioNet:
    checkpoint = torch.load(path, map_location=device)
    model = GATPortfolioNet(
        n_stocks    = checkpoint["n_stocks"],
        n_features  = checkpoint["features_per_bar"],
        d_model     = checkpoint["d_model"],
        n_layers_tf = checkpoint["n_layers_tf"],
        n_heads_gat = checkpoint["n_heads_gat"],
        gat_layers  = checkpoint["gat_layers"],
        dropout     = checkpoint["dropout"],
    )
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)
    model.eval()
    print(f"Model loaded from {path}")
    print(f"  Saved at    : {checkpoint['saved_at']}")
    print(f"  Tickers     : {checkpoint['tickers']}")
    print(f"  Candle freq : {checkpoint['candle_freq']}")
    print(f"  Lookback    : {checkpoint['lookback']}")
    return model


# --- Call in main() after training, before backtest: ---
#model = train(model, train_df, TICKERS, device=device)
save_model(model, path="gat_portfolio.pth")

# --- To reload later: ---
# model = load_model("gat_portfolio.pth", device=device)

Model saved to gat_portfolio.pth


In [ ]:
def model_simulation(
    model_path: str = "gat_portfolio.pth",
    initial_capital: float = 100_000.0,
    device: str = "cpu",
) -> pd.DataFrame:
    """
    Loads the saved model and simulates weekly rebalancing through 2017.
    Starts with equal-weight allocation and adjusts every week per model output.
    Tracks capital, weights, turnover, and transaction costs.
    """
    # ── Load model ────────────────────────────────────────────────────────
    model      = load_model(model_path, device=device)
    checkpoint = torch.load(model_path, map_location=device)
    tickers    = checkpoint["tickers"]
    lookback   = checkpoint["lookback"]
    n          = len(tickers)

    # ── Fetch data ────────────────────────────────────────────────────────
    # Need data before 2017 to fill the lookback window (52 weekly candles)
    # 52 weeks lookback + buffer → start from early 2015
    df = fetch_price_data(
        tickers,
        start = "2015-01-01",
        end   = "2017-12-31",
        freq  = CandleFreq[checkpoint["candle_freq"]],
    )
    df = df.ffill().dropna()

    # Isolate the 2017 weekly bars
    sim_df    = df[df.index.year == 2017]
    sim_dates = sim_df.index.tolist()

    assert len(sim_dates) >= 2, "No 2017 weekly bars found."
    print(f"\n2017 Simulation | {len(sim_dates)} weekly bars | "
          f"{sim_dates[0].date()} -> {sim_dates[-1].date()}")
    print(f"Tickers  : {tickers}")
    print(f"Capital  : ${initial_capital:,.0f}")
    print(f"{'─'*70}")

    # ── State ─────────────────────────────────────────────────────────────
    history       = []              # regret memory — empty at start of year
    capital       = initial_capital
    prev_weights  = np.ones(n) / n  # start equal-weight
    records       = []

    for i, date in enumerate(sim_dates[:-1]):   # last bar has no next week
        # Global index of this date in the full df (needed for feature tensor)
        global_idx = df.index.get_loc(date)

        # Need at least lookback + 4 bars before this point
        if global_idx < lookback + 4:
            print(f"  [{date.date()}] Skipping — not enough lookback history yet.")
            continue

        # ── Build inputs ──────────────────────────────────────────────────
        x      = build_feature_tensor(df, tickers, global_idx, lookback).to(device)
        ei, ea = build_correlation_graph(df, tickers, global_idx, lookback)
        ei, ea = ei.to(device), ea.to(device)

        # ── Model inference ───────────────────────────────────────────────
        model.eval()
        with torch.no_grad():
            weights_tensor = model(x, ei, ea, history=history, device=device)

        if torch.isnan(weights_tensor).any():
            print(f"  [{date.date()}] NaN weights — holding previous allocation.")
            new_weights = prev_weights
        else:
            new_weights = weights_tensor.cpu().numpy()

        # ── Turnover & transaction cost (5bps per unit traded) ────────────
        turnover      = float(np.abs(new_weights - prev_weights).sum())
        tx_cost_rate  = 0.0005              # 5 basis points per side
        tx_cost       = capital * turnover * tx_cost_rate

        # ── Realised return (this week → next week) ───────────────────────
        next_date  = sim_dates[i + 1]
        cur_price  = df["Close"][tickers].loc[date].values.astype(np.float64)
        next_price = df["Close"][tickers].loc[next_date].values.astype(np.float64)
        ret_vec    = (next_price - cur_price) / (cur_price + 1e-8)
        port_ret   = float((new_weights * ret_vec).sum())

        # ── Update capital ────────────────────────────────────────────────
        capital = (capital - tx_cost) * (1 + port_ret)

        # ── Update regret history ─────────────────────────────────────────
        history.append((
            torch.tensor(new_weights, dtype=torch.float32),
            torch.tensor(ret_vec,     dtype=torch.float32),
        ))
        if len(history) > model.memory.memory_len:
            history.pop(0)

        records.append({
            "date":           date,
            "capital":        capital,
            "weekly_ret":     port_ret,
            "turnover":       turnover,
            "tx_cost":        tx_cost,
            **{t: float(w) for t, w in zip(tickers, new_weights)},
        })

        # ── Weekly print ──────────────────────────────────────────────────
        top2 = sorted(zip(tickers, new_weights), key=lambda x: -x[1])[:2]
        top2_str = "  ".join(f"{t}={w*100:.1f}%" for t, w in top2)
        print(f"  [{date.date()}]  ret={port_ret*100:+5.2f}%  "
              f"capital=${capital:>10,.0f}  "
              f"top2: {top2_str}  "
              f"turnover={turnover*100:.1f}%")

        prev_weights = new_weights

    # ── Build results ──────────────────────────────────────────────────────
    result = pd.DataFrame(records).set_index("date")
    result["cumulative_ret"] = (1 + result["weekly_ret"]).cumprod() - 1

    r   = result["weekly_ret"]
    ann = (1 + r.mean()) ** 52 - 1
    sh  = r.mean() / (r.std() + 1e-8) * math.sqrt(52)
    pv  = result["cumulative_ret"] + 1
    mdd = ((pv.cummax() - pv) / pv.cummax()).max()

    total_tx = result["tx_cost"].sum()
    avg_turn = result["turnover"].mean()

    print(f"\n{'─'*70}")
    print(f"2017 Simulation Results")
    print(f"  Start capital     : ${initial_capital:>12,.2f}")
    print(f"  End capital       : ${capital:>12,.2f}")
    print(f"  Total return      : {result['cumulative_ret'].iloc[-1]*100:+.2f}%")
    print(f"  Annualised return : {ann*100:+.2f}%")
    print(f"  Sharpe            : {sh:.3f}")
    print(f"  Max drawdown      : {mdd*100:.2f}%")
    print(f"  Total tx costs    : ${total_tx:,.2f}")
    print(f"  Avg weekly turnover: {avg_turn*100:.1f}%")

    return result
def buy_and_hold(
    initial_capital: float = 100_000.0,
) -> pd.DataFrame:
    """
    Equal-weight buy & hold through 2017 — no rebalancing, no transaction costs.
    Uses identical dates and price data as run_2017_simulation() for direct comparison.
    """
    tickers = TICKERS
    n       = len(tickers)

    # ── Fetch same data as simulation ─────────────────────────────────────
    df = fetch_price_data(
        tickers,
        start = "2015-01-01",
        end   = "2017-12-31",
        freq  = CANDLE_FREQ,
    )
    df = df.ffill().dropna()

    sim_df    = df[df.index.year == 2017]
    sim_dates = sim_df.index.tolist()

    assert len(sim_dates) >= 2, "No 2017 weekly bars found."
    print(f"\n2017 Buy & Hold | {len(sim_dates)} weekly bars | "
          f"{sim_dates[0].date()} -> {sim_dates[-1].date()}")
    print(f"Tickers  : {tickers}")
    print(f"Capital  : ${initial_capital:,.0f}")
    print(f"{'─'*70}")

    # ── Entry: equal weight on first 2017 bar ─────────────────────────────
    entry_price = df["Close"][tickers].loc[sim_dates[0]].values.astype(np.float64)
    shares      = (initial_capital / n) / entry_price   # fixed for entire year

    capital      = initial_capital
    records      = []

    for i, date in enumerate(sim_dates[:-1]):
        next_date  = sim_dates[i + 1]
        cur_price  = df["Close"][tickers].loc[date].values.astype(np.float64)
        next_price = df["Close"][tickers].loc[next_date].values.astype(np.float64)

        # Portfolio value at this bar
        position_vals = shares * cur_price
        port_val      = position_vals.sum()

        # Drifted weights (informational — these change as prices diverge)
        weights   = position_vals / port_val

        # Return this week
        ret_vec   = (next_price - cur_price) / (cur_price + 1e-8)
        port_ret  = float((weights * ret_vec).sum())

        # Update capital
        capital   = port_val * (1 + port_ret)

        records.append({
            "date":        date,
            "capital":     capital,
            "weekly_ret":  port_ret,
            "turnover":    0.0,      # no rebalancing ever
            "tx_cost":     0.0,
            **{t: float(w) for t, w in zip(tickers, weights)},
        })

        # ── Weekly print ──────────────────────────────────────────────────
        top2 = sorted(zip(tickers, weights), key=lambda x: -x[1])[:2]
        top2_str = "  ".join(f"{t}={w*100:.1f}%" for t, w in top2)
        print(f"  [{date.date()}]  ret={port_ret*100:+5.2f}%  "
              f"capital=${capital:>10,.0f}  "
              f"top2: {top2_str}")

    # ── Build results ──────────────────────────────────────────────────────
    result = pd.DataFrame(records).set_index("date")
    result["cumulative_ret"] = (1 + result["weekly_ret"]).cumprod() - 1

    r   = result["weekly_ret"]
    ann = (1 + r.mean()) ** 52 - 1
    sh  = r.mean() / (r.std() + 1e-8) * math.sqrt(52)
    pv  = result["cumulative_ret"] + 1
    mdd = ((pv.cummax() - pv) / pv.cummax()).max()

    # Final drifted weights
    final_price  = df["Close"][tickers].loc[sim_dates[-1]].values.astype(np.float64)
    final_vals   = shares * final_price
    final_w      = final_vals / final_vals.sum()

    print(f"\n{'─'*70}")
    print(f"2017 Buy & Hold Results")
    print(f"  Start capital     : ${initial_capital:>12,.2f}")
    print(f"  End capital       : ${capital:>12,.2f}")
    print(f"  Total return      : {result['cumulative_ret'].iloc[-1]*100:+.2f}%")
    print(f"  Annualised return : {ann*100:+.2f}%")
    print(f"  Sharpe            : {sh:.3f}")
    print(f"  Max drawdown      : {mdd*100:.2f}%")
    print(f"\n  Weight drift (entry -> exit):")
    for t in tickers:
        print(f"    {t:<6}  {100/n:.1f}%  ->  {final_w[tickers.index(t)]*100:.1f}%")

    return result


# ── Call both and compare ─────────────────────────────────────────────────
if __name__ == "__main__":
    device = "cuda" if torch.cuda.is_available() else "cpu"

    sim_result = model_simulation(
        model_path      = "gat_portfolio.pth",
        initial_capital = 100_000.0,
        device          = device,
    )

    bh_result = buy_and_hold(
        initial_capital = 100_000.0,
    )

    # ── Side-by-side ──────────────────────────────────────────────────────
    print(f"\n{'='*70}")
    print(f"{'FINAL COMPARISON':^70}")
    print(f"{'='*70}")
    print(f"  {'Metric':<22}  {'GAT Model':>14}  {'Buy & Hold':>14}")
    print(f"  {'─'*22}  {'─'*14}  {'─'*14}")

    def stats(r):
        ann = (1 + r["weekly_ret"].mean()) ** 52 - 1
        sh  = r["weekly_ret"].mean() / (r["weekly_ret"].std() + 1e-8) * math.sqrt(52)
        pv  = r["cumulative_ret"] + 1
        mdd = ((pv.cummax() - pv) / pv.cummax()).max()
        tot = r["cumulative_ret"].iloc[-1]
        return tot, ann, sh, mdd

    m_tot, m_ann, m_sh, m_mdd = stats(sim_result)
    b_tot, b_ann, b_sh, b_mdd = stats(bh_result)

    print(f"  {'Total return':<22}  {m_tot*100:>+13.2f}%  {b_tot*100:>+13.2f}%")
    print(f"  {'Annualised return':<22}  {m_ann*100:>+13.2f}%  {b_ann*100:>+13.2f}%")
    print(f"  {'Sharpe':<22}  {m_sh:>14.3f}  {b_sh:>14.3f}")
    print(f"  {'Max drawdown':<22}  {m_mdd*100:>13.2f}%  {b_mdd*100:>13.2f}%")
    print(f"  {'End capital':<22}  ${sim_result['capital'].iloc[-1]:>13,.2f}  "
          f"${bh_result['capital'].iloc[-1]:>13,.2f}")
    print(f"  {'Total tx costs':<22}  ${sim_result['tx_cost'].sum():>13,.2f}  "
          f"${'0.00':>13}")

Model loaded from gat_portfolio.pth
  Saved at    : 2026-03-20 15:08:05
  Tickers     : ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA', 'JPM', 'V', 'UNH']
  Candle freq : WEEKLY
  Lookback    : 52
Candle freq : WEEKLY | bars: 157 | lookback: 52 candles | rebalance every: 1 candle(s)

2017 Simulation | 52 weekly bars | 2017-01-06 -> 2017-12-29
Tickers  : ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA', 'JPM', 'V', 'UNH']
Capital  : $100,000
──────────────────────────────────────────────────────────────────────
  [2017-01-06]  ret=+1.07%  capital=$   101,023  top2: GOOGL=25.2%  TSLA=25.2%  turnover=91.3%
  [2017-01-13]  ret=+0.67%  capital=$   101,705  top2: GOOGL=25.3%  TSLA=25.3%  turnover=0.3%
  [2017-01-20]  ret=+2.93%  capital=$   104,686  top2: GOOGL=25.2%  TSLA=25.2%  turnover=3.0%
  [2017-01-27]  ret=-0.28%  capital=$   104,387  top2: GOOGL=25.2%  TSLA=25.2%  turnover=4.4%
  [2017-02-03]  ret=+2.40%  capital=$   106,885  top2: TSLA=25.0%  V=25.0%  turnover=7.4%